# Social Backlash Risk Pipeline with Qwen3-8B

This notebook implements a two-step risk analysis flow for short public SNS posts.

Pipeline:
1. User Input
2. Preprocessing
3. Tokenization
4. Embedding (sentence + contextual)
5. First-pass Risk Classification (rules + semantic similarity)
6. Context-based Analysis with Qwen3-8B
7. Rewrite Suggestion
8. Final Output

Scoring idea:
`final risk = heuristic score + semantic score + optional Qwen correction`


In [ ]:
# Run once if needed.
# %pip install -q pandas scikit-learn sentence-transformers torch transformers accelerate ipykernel


In [ ]:
import json
import math
import os
import re
import urllib.error
import urllib.request
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMER_AVAILABLE = True
except ImportError:
    SentenceTransformer = None
    SENTENCE_TRANSFORMER_AVAILABLE = False

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    torch = None
    AutoModelForCausalLM = None
    AutoTokenizer = None
    TRANSFORMERS_AVAILABLE = False

pd.set_option("display.max_colwidth", 120)

def torch_cuda_available() -> bool:
    return bool(torch is not None and hasattr(torch, "cuda") and torch.cuda.is_available())


def get_runtime_device() -> str:
    return "cuda" if torch_cuda_available() else "cpu"


def get_gpu_memory_gb() -> float:
    if not torch_cuda_available():
        return 0.0
    try:
        total_bytes = torch.cuda.get_device_properties(0).total_memory
        return round(total_bytes / (1024 ** 3), 2)
    except Exception:
        return 0.0


RUNTIME_DEVICE = get_runtime_device()
GPU_MEMORY_GB = get_gpu_memory_gb()
MIN_QWEN_TRANSFORMERS_VRAM_GB = 14.0

QWEN_BACKEND = "ollama"  # auto | ollama | transformers | disabled
QWEN_OLLAMA_MODEL = "qwen3:8b"
QWEN_OLLAMA_URL = "http://localhost:11434/api/generate"
QWEN_TRANSFORMERS_MODEL = "Qwen/Qwen3-8B"
SOCIAL_CONTEXT = "Instagram / Public"
AUDIENCE_PROFILE = (
    "Public Instagram audience reading a short opinion post. Readers include casual followers, "
    "fans of the target, neutral bystanders, and people sensitive to sarcasm, dunking, or sweeping claims."
)

DIMENSION_ORDER = [
    "Aggression",
    "Group Generalization",
    "Sarcasm / Mockery",
    "Overconfident Judgment",
    "Context Inappropriateness",
    "Misinterpretability",
    "Norm Violation",
]

DIMENSION_DESCRIPTIONS = {
    "Aggression": "Level of direct aggression or harsh criticism",
    "Group Generalization": "Degree of broad judgment about an entire group",
    "Sarcasm / Mockery": "Intensity of sarcasm, mockery, or derision",
    "Overconfident Judgment": "Degree to which a personal opinion is framed like an absolute fact",
    "Context Inappropriateness": "Likelihood of appearing inappropriate in a public SNS context",
    "Misinterpretability": "Likelihood that the intent could be distorted or read as hostile",
    "Norm Violation": "Tendency toward discriminatory, dehumanizing, or norm-violating language",
}

DIMENSION_WEIGHTS = {
    "Aggression": 1.35,
    "Group Generalization": 1.2,
    "Sarcasm / Mockery": 1.1,
    "Overconfident Judgment": 0.95,
    "Context Inappropriateness": 1.15,
    "Misinterpretability": 1.0,
    "Norm Violation": 1.4,
}

CATEGORY_LABELS = {
    "Aggression": "Aggressive phrasing",
    "Group Generalization": "Sweeping generalization",
    "Sarcasm / Mockery": "Mocking tone",
    "Overconfident Judgment": "Absolute judgment",
    "Context Inappropriateness": "Public-context mismatch",
    "Misinterpretability": "Easy to misread",
    "Norm Violation": "Norm boundary risk",
}


In [ ]:
IMPORTANT_EMOJIS = {"lol", "lmao", "rofl", "smh", "ikr", "fr", "tbh", "wtf", "omg", "^^", ":)", ":(", "haha"}
PUBLIC_CONTEXT_HINTS = {"public", "instagram", "x", "twitter", "community", "forum", "tiktok", "comment section"}
VAGUE_TARGET_HINTS = {"they", "them", "people", "everyone", "those", "that", "this"}
AGGRESSIVE_TERMS = {"trash", "garbage", "pathetic", "stupid", "dumb", "awful", "hate", "disgusting", "worst", "sucks", "terrible"}
GROUP_CUES = {"all", "every", "everyone", "nobody", "always", "these people", "that group"}
SARCASM_CUES = {"lol", "lmao", "sure", "yeah right", "as if", "clown", "cute", "okayyy"}
NORM_VIOLATION_CUES = {"subhuman", "animal", "vermin", "disgusting people", "those types"}
DIRECT_ATTACK_PATTERNS = [r"\bis trash\b", r"\bare trash\b", r"\bis garbage\b", r"\bare garbage\b", r"\bwhat a joke\b"]
DISMISSIVE_PATTERNS = [r"\blol\b", r"\blmao\b", r"\byeah right\b", r"\bas if\b"]
HEDGE_PATTERNS = [r"\bi think\b", r"\bi feel\b", r"\bmaybe\b", r"\bperhaps\b", r"\bkind of\b", r"\bit seems\b"]

DIMENSION_PROTOTYPES = {
    "Aggression": [
        "This sounds openly hostile and harsh.",
        "The post attacks the target in a blunt and aggressive way.",
        "This wording feels insulting and confrontational.",
    ],
    "Group Generalization": [
        "The post makes a sweeping claim about a whole group.",
        "This sounds like a broad generalization about many people.",
        "The wording treats a group as if everyone in it is the same.",
    ],
    "Sarcasm / Mockery": [
        "The tone sounds sarcastic, dismissive, or mocking.",
        "This post seems written to ridicule someone.",
        "The wording has a sneering or eye-rolling tone.",
    ],
    "Overconfident Judgment": [
        "The post presents a personal opinion like an unquestionable fact.",
        "This sounds overly certain and leaves little room for nuance.",
        "The wording feels absolute rather than tentative.",
    ],
    "Context Inappropriateness": [
        "This could feel inappropriate in a public social media setting.",
        "The same statement may draw criticism when posted publicly.",
        "This wording may not fit a broad online audience.",
    ],
    "Misinterpretability": [
        "The message is easy to misunderstand and may sound harsher than intended.",
        "This post is ambiguous enough that readers may infer a hostile meaning.",
        "The target or intent is unclear, which raises the chance of misreading.",
    ],
    "Norm Violation": [
        "The tone risks sounding demeaning, dehumanizing, or socially unacceptable.",
        "This wording may cross a social norm boundary.",
        "The post could be read as attacking someone's dignity or status.",
    ],
}

@dataclass
class PipelineOutput:
    context: str
    recommended_context: str
    audience_profile: str
    original_text: str
    normalized_text: str
    tokens: List[str]
    embedding_model: str
    sentence_embedding_preview: List[float]
    contextual_embedding_preview: List[float]
    heuristic_dimensions: Dict[str, int]
    llm_adjustments: Dict[str, int]
    final_dimensions: Dict[str, int]
    heuristic_risk_score: int
    llm_risk_score: Optional[int]
    final_risk_score: int
    issue_categories: List[str]
    special_situations: List[str]
    problematic_phrases: List[str]
    explanation_points: List[str]
    llm_backend: str
    llm_prompt: str
    rewrite_suggestion: str


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def preserve_and_clean_special_characters(text: str) -> str:
    text = text.replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    text = re.sub(r"[^\w\s!?.,'@#:/()~+\-]", " ", text)
    text = re.sub(r"([!?.,~]){3,}", r"\1\1", text)
    return normalize_whitespace(text)


def reduce_repeated_characters(text: str, keep: int = 2) -> str:
    return re.sub(r"(.)\1{2,}", lambda match: match.group(1) * keep, text)


def preprocess_text(text: str) -> str:
    text = normalize_whitespace(text)
    text = preserve_and_clean_special_characters(text)
    text = reduce_repeated_characters(text)
    return text


def tokenize_text(text: str) -> List[str]:
    token_pattern = r"#[\w_]+|@[\w_]+|[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[!?.,]+"
    return re.findall(token_pattern, text.lower())


def clipped_dimension(value: float, low: float = 0.38, high: float = 0.62) -> int:
    if value >= high:
        return 2
    if value >= low:
        return 1
    return 0


def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    norm_a = float(np.linalg.norm(vec_a))
    norm_b = float(np.linalg.norm(vec_b))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / (norm_a * norm_b))


In [ ]:
class HybridEmbedder:
    def __init__(self, seed_texts: Optional[List[str]] = None):
        prototype_texts = [item for values in DIMENSION_PROTOTYPES.values() for item in values]
        self.seed_texts = seed_texts or [
            "this place is trash lol",
            "everyone here is so annoying",
            "i disagree but i may be wrong",
            "the service was not my style",
            "why are people always like this",
        ] + prototype_texts
        self.tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
        self.tfidf.fit(self.seed_texts)
        self.model_name = "tfidf-fallback"
        self.sentence_model = None

        if SENTENCE_TRANSFORMER_AVAILABLE:
            try:
                sentence_device = "cuda" if RUNTIME_DEVICE == "cuda" else "cpu"
                self.sentence_model = SentenceTransformer("all-MiniLM-L6-v2", device=sentence_device)
                self.model_name = f"sentence-transformers/all-MiniLM-L6-v2 [{sentence_device}]"
            except Exception:
                self.sentence_model = None

    def encode_sentence(self, text: str) -> np.ndarray:
        if self.sentence_model is not None:
            return np.asarray(self.sentence_model.encode(text), dtype=float)
        return self.tfidf.transform([text]).toarray()[0]

    def encode_contextual(self, text: str, context: str, audience_profile: str) -> np.ndarray:
        contextual_text = f"Context: {context}\nAudience: {audience_profile}\nPost: {text}"
        return self.encode_sentence(contextual_text)


class SemanticRiskScorer:
    def __init__(self, embedder: HybridEmbedder):
        self.embedder = embedder
        self.prototype_embeddings = {
            name: [self.embedder.encode_sentence(example) for example in examples]
            for name, examples in DIMENSION_PROTOTYPES.items()
        }

    def prototype_score(self, sentence_embedding: np.ndarray, contextual_embedding: np.ndarray, dimension: str) -> float:
        similarities = []
        for prototype_embedding in self.prototype_embeddings[dimension]:
            sentence_score = cosine_similarity(sentence_embedding, prototype_embedding)
            contextual_score = cosine_similarity(contextual_embedding, prototype_embedding)
            similarities.append((0.55 * sentence_score) + (0.45 * contextual_score))
        return max(similarities) if similarities else 0.0


def measure_surface_signals(text: str, tokens: List[str], context: str) -> Dict[str, float]:
    lowered = text.lower()
    exclamations = min(text.count("!"), 3) / 3
    question_bursts = min(text.count("?"), 3) / 3
    repeated_punctuation = 1.0 if re.search(r"([!?.,~])\1", text) else 0.0
    emoji_density = min(sum(token in IMPORTANT_EMOJIS for token in tokens), 3) / 3
    hedge_density = min(sum(bool(re.search(pattern, lowered)) for pattern in HEDGE_PATTERNS), 2) / 2
    public_context = 1.0 if any(hint in context.lower() for hint in PUBLIC_CONTEXT_HINTS) else 0.0
    short_post = 1.0 if len(tokens) <= 6 else 0.0
    vague_target = 1.0 if any(token in VAGUE_TARGET_HINTS for token in tokens) else 0.0
    aggressive_lexicon = min(sum(term in lowered for term in AGGRESSIVE_TERMS), 3) / 3
    group_lexicon = min(sum(term in lowered for term in GROUP_CUES), 3) / 3
    sarcasm_lexicon = min(sum(term in lowered for term in SARCASM_CUES), 3) / 3
    norm_lexicon = min(sum(term in lowered for term in NORM_VIOLATION_CUES), 2) / 2
    direct_attack = 1.0 if any(re.search(pattern, lowered) for pattern in DIRECT_ATTACK_PATTERNS) else 0.0
    dismissive_marker = 1.0 if any(re.search(pattern, lowered) for pattern in DISMISSIVE_PATTERNS) else 0.0
    sentiment_targeting = 1.0 if re.search(r"\b(this|that|it|they|he|she|you)\b.+\b(is|are)\b.+\b(" + "|".join(sorted(AGGRESSIVE_TERMS)) + r")\b", lowered) else 0.0

    return {
        "exclamations": exclamations,
        "question_bursts": question_bursts,
        "repeated_punctuation": repeated_punctuation,
        "emoji_density": emoji_density,
        "hedge_density": hedge_density,
        "public_context": public_context,
        "short_post": short_post,
        "vague_target": vague_target,
        "aggressive_lexicon": aggressive_lexicon,
        "group_lexicon": group_lexicon,
        "sarcasm_lexicon": sarcasm_lexicon,
        "norm_lexicon": norm_lexicon,
        "direct_attack": direct_attack,
        "dismissive_marker": dismissive_marker,
        "sentiment_targeting": sentiment_targeting,
    }


def first_pass_risk_classification(
    text: str,
    tokens: List[str],
    context: str,
    sentence_embedding: np.ndarray,
    contextual_embedding: np.ndarray,
    scorer: SemanticRiskScorer,
) -> Dict[str, int]:
    semantic_scores = {
        name: scorer.prototype_score(sentence_embedding, contextual_embedding, name)
        for name in DIMENSION_ORDER
    }
    surface = measure_surface_signals(text, tokens, context)

    aggression = semantic_scores["Aggression"] + 0.28 * surface["aggressive_lexicon"] + 0.32 * surface["direct_attack"] + 0.18 * surface["sentiment_targeting"] + 0.08 * surface["exclamations"] + 0.08 * surface["repeated_punctuation"]
    group = semantic_scores["Group Generalization"] + 0.18 * surface["group_lexicon"] + 0.10 * surface["vague_target"]
    sarcasm = semantic_scores["Sarcasm / Mockery"] + 0.18 * surface["sarcasm_lexicon"] + 0.18 * surface["dismissive_marker"] + 0.08 * surface["emoji_density"] + 0.05 * surface["question_bursts"]
    overconfidence = semantic_scores["Overconfident Judgment"] - 0.12 * surface["hedge_density"] + 0.06 * (1.0 - surface["hedge_density"])
    context_signal = semantic_scores["Context Inappropriateness"] + 0.15 * surface["public_context"] + 0.08 * surface["dismissive_marker"] + 0.05 * surface["short_post"]
    misread = semantic_scores["Misinterpretability"] + 0.10 * surface["short_post"] + 0.10 * surface["vague_target"] + 0.06 * surface["dismissive_marker"]
    norm = semantic_scores["Norm Violation"] + 0.18 * surface["norm_lexicon"] + 0.05 * surface["public_context"]
    # Hybrid rule: hedging softens certainty, but not the hostility of phrases like 'is trash lol'.
    if surface["direct_attack"] and surface["dismissive_marker"]:
        aggression = max(aggression, 0.58 - 0.22 * surface["hedge_density"])
        sarcasm = max(sarcasm, 0.52)
        context_signal = max(context_signal, 0.46 - 0.08 * surface["hedge_density"])
    elif surface["direct_attack"]:
        aggression = max(aggression, 0.48 - 0.18 * surface["hedge_density"])
    elif surface["aggressive_lexicon"] and surface["dismissive_marker"]:
        aggression = max(aggression, 0.40 - 0.12 * surface["hedge_density"])
        sarcasm = max(sarcasm, 0.42)

    aggression_level = clipped_dimension(aggression)
    group_level = clipped_dimension(group)
    sarcasm_level = clipped_dimension(sarcasm)
    overconfidence_level = clipped_dimension(overconfidence)
    context_level = clipped_dimension(context_signal)
    misread_level = clipped_dimension(misread)
    norm_level = clipped_dimension(norm)

    # Hedge phrases such as 'I think' should soften certainty a bit, but not erase a direct insult.
    if surface["hedge_density"] >= 0.5 and surface["direct_attack"]:
        aggression_level = max(1, aggression_level - 1)
        context_level = max(0, context_level - 1)

    return {
        "Aggression": aggression_level,
        "Group Generalization": group_level,
        "Sarcasm / Mockery": sarcasm_level,
        "Overconfident Judgment": overconfidence_level,
        "Context Inappropriateness": context_level,
        "Misinterpretability": misread_level,
        "Norm Violation": norm_level,
    }


def calculate_weighted_risk_score(risk_dimensions: Dict[str, int]) -> int:
    weighted_sum = sum(risk_dimensions[name] * DIMENSION_WEIGHTS[name] for name in DIMENSION_ORDER)
    max_sum = sum(2 * DIMENSION_WEIGHTS[name] for name in DIMENSION_ORDER)
    score = round((weighted_sum / max_sum) * 100)
    return int(np.clip(score, 0, 100))


def derive_issue_categories(risk_dimensions: Dict[str, int]) -> List[str]:
    categories = [CATEGORY_LABELS[name] for name in DIMENSION_ORDER if risk_dimensions[name] >= 1]
    return categories or ["Low explicit risk"]


In [ ]:
def extract_first_json_block(text: str) -> Optional[dict]:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


def build_qwen_prompt(
    context: str,
    audience_profile: str,
    original_text: str,
    normalized_text: str,
    tokens: List[str],
    first_pass_dimensions: Dict[str, int],
) -> str:
    first_pass_lines = "\n".join(f"- {name}: {score}" for name, score in first_pass_dimensions.items())
    return f"""
You are a social-backlash analyst for short public SNS posts.

Goal: estimate the probability that a post receives criticism, backlash, quote-tweeting, or negative replies.
Focus on social backlash, not only toxicity.

Context: {context}
Audience profile: {audience_profile}
Original text: {original_text}
Normalized text: {normalized_text}
Tokens: {tokens}

First-pass scores:
{first_pass_lines}

Return ONLY valid JSON with this schema:
{{
  "backlash_probability": 0,
  "adjustments": {{
    "Aggression": 0,
    "Group Generalization": 0,
    "Sarcasm / Mockery": 0,
    "Overconfident Judgment": 0,
    "Context Inappropriateness": 0,
    "Misinterpretability": 0,
    "Norm Violation": 0
  }},
  "special_situations": ["string"],
  "problematic_phrases": ["string"],
  "explanations": ["string", "string", "string"],
  "rewrite": "string"
}}

Rules:
- Adjustments must be only -1, 0, or 1.
- Keep explanations concrete and audience-facing.
- Mention if the post sounds dismissive, snarky, unfair, too absolute, or easy to misread.
- Use the public audience context when deciding backlash risk.
- The rewrite should keep the speaker's intent but lower backlash risk.
""".strip()


class QwenContextAnalyzer:
    def __init__(self, backend: str = "auto"):
        self.backend = backend
        self._transformers_tokenizer = None
        self._transformers_model = None
        self._transformers_reason = ""

    def _can_use_ollama(self) -> bool:
        req = urllib.request.Request(QWEN_OLLAMA_URL, method="POST")
        body = json.dumps({"model": QWEN_OLLAMA_MODEL, "prompt": "ping", "stream": False}).encode("utf-8")
        try:
            with urllib.request.urlopen(req, data=body, timeout=2) as response:
                return response.status == 200
        except Exception:
            return False

    def _generate_with_ollama(self, prompt: str) -> Optional[dict]:
        payload = {
            "model": QWEN_OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "format": "json",
            "options": {"temperature": 0.2},
        }
        req = urllib.request.Request(QWEN_OLLAMA_URL, method="POST")
        try:
            with urllib.request.urlopen(req, data=json.dumps(payload).encode("utf-8"), timeout=120) as response:
                data = json.loads(response.read().decode("utf-8"))
                return extract_first_json_block(data.get("response", ""))
        except (urllib.error.URLError, TimeoutError, json.JSONDecodeError):
            return None

    def _load_transformers(self) -> bool:
        if not TRANSFORMERS_AVAILABLE:
            self._transformers_reason = "transformers-not-installed"
            return False
        if self._transformers_model is not None and self._transformers_tokenizer is not None:
            return True
        if not torch_cuda_available():
            self._transformers_reason = "cuda-not-available"
            return False
        if GPU_MEMORY_GB < MIN_QWEN_TRANSFORMERS_VRAM_GB:
            self._transformers_reason = f"insufficient-vram-{GPU_MEMORY_GB}GB"
            return False
        try:
            self._transformers_tokenizer = AutoTokenizer.from_pretrained(QWEN_TRANSFORMERS_MODEL)
            model_kwargs = {
                "torch_dtype": torch.float16,
                "device_map": "auto",
                "low_cpu_mem_usage": True,
            }
            self._transformers_model = AutoModelForCausalLM.from_pretrained(QWEN_TRANSFORMERS_MODEL, **model_kwargs)
            return True
        except Exception:
            self._transformers_model = None
            self._transformers_tokenizer = None
            self._transformers_reason = "transformers-load-failed"
            return False

    def _generate_with_transformers(self, prompt: str) -> Optional[dict]:
        if not self._load_transformers():
            return None
        try:
            messages = [{"role": "user", "content": prompt}]
            text = self._transformers_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = self._transformers_tokenizer(text, return_tensors="pt")
            if hasattr(self._transformers_model, "device"):
                inputs = {key: value.to(self._transformers_model.device) for key, value in inputs.items()}
            output = self._transformers_model.generate(**inputs, max_new_tokens=400, do_sample=False)
            generated = output[0][inputs["input_ids"].shape[-1]:]
            decoded = self._transformers_tokenizer.decode(generated, skip_special_tokens=True)
            return extract_first_json_block(decoded)
        except Exception:
            return None

    def analyze(self, prompt: str) -> Dict[str, object]:
        if self.backend == "disabled":
            return {"backend": "disabled", "result": None}
        if self.backend in {"auto", "ollama"} and self._can_use_ollama():
            result = self._generate_with_ollama(prompt)
            if result is not None:
                return {"backend": "ollama", "result": result}
        if self.backend in {"auto", "transformers"}:
            result = self._generate_with_transformers(prompt)
            if result is not None:
                return {"backend": "transformers", "result": result}
            if self.backend == "transformers" and self._transformers_reason:
                return {"backend": f"transformers-skipped:{self._transformers_reason}", "result": None}
        return {"backend": "unavailable", "result": None}


def sanitize_llm_adjustments(payload: Optional[dict]) -> Dict[str, int]:
    adjustments = {name: 0 for name in DIMENSION_ORDER}
    if not payload:
        return adjustments
    raw = payload.get("adjustments", {})
    for name in DIMENSION_ORDER:
        value = raw.get(name, 0)
        if isinstance(value, (int, float)):
            adjustments[name] = int(np.clip(round(value), -1, 1))
    return adjustments


def soften_expression(text: str) -> str:
    softened = preserve_and_clean_special_characters(text).strip(" .!?")
    if not softened:
        return "I would phrase this more neutrally."
    softened = softened[0].lower() + softened[1:] if len(softened) > 1 else softened.lower()
    if not re.search(r"\bi\b", softened):
        softened = f"I think {softened}"
    softened = softened.rstrip(".") + "."
    return softened[0].upper() + softened[1:]


def build_recommended_context(text: str, final_dimensions: Dict[str, int], context: str) -> str:
    revised = preserve_and_clean_special_characters(text)
    revised = re.sub(r"\bis trash\b", "wasn't my preference", revised, flags=re.IGNORECASE)
    revised = re.sub(r"\bare trash\b", "weren't a good fit for me", revised, flags=re.IGNORECASE)
    revised = re.sub(r"\bis garbage\b", "was disappointing", revised, flags=re.IGNORECASE)
    revised = re.sub(r"\blol\b|\blmao\b", "", revised, flags=re.IGNORECASE)
    revised = re.sub(r"\s+", " ", revised).strip(" .!?")

    if final_dimensions["Overconfident Judgment"] >= 1 and not re.search(r"\bI think\b|\bin my opinion\b", revised, flags=re.IGNORECASE):
        revised = f"I think {revised}" if revised else "I think this wasn't my preference"
    elif not revised.lower().startswith("i think") and not revised.lower().startswith("in my opinion"):
        revised = f"I think {revised}" if revised else "I think this wasn't my preference"

    if final_dimensions["Misinterpretability"] >= 1 and "because" not in revised.lower():
        revised += " because it did not match what I was expecting"

    if any(hint in context.lower() for hint in PUBLIC_CONTEXT_HINTS):
        revised = revised.rstrip(".") + "."

    return revised or "I think this wasn't my preference."


def detect_problematic_phrases(text: str) -> List[str]:
    lowered = text.lower()
    phrases = []
    for pattern in DIRECT_ATTACK_PATTERNS:
        match = re.search(pattern, lowered)
        if match:
            phrases.append(text[match.start():match.end()])
    for pattern in DISMISSIVE_PATTERNS:
        match = re.search(pattern, lowered)
        if match:
            phrases.append(text[match.start():match.end()])
    return list(dict.fromkeys(phrases))


def detect_special_situations(final_dimensions: Dict[str, int], text: str, context: str) -> List[str]:
    situations = []
    lowered = text.lower()
    if final_dimensions["Aggression"] >= 1 and any(hint in context.lower() for hint in PUBLIC_CONTEXT_HINTS):
        situations.append("public negative review tone")
    if final_dimensions["Sarcasm / Mockery"] >= 1 and any(token in lowered for token in SARCASM_CUES):
        situations.append("dismissive humor / dunking risk")
    if final_dimensions["Misinterpretability"] >= 1 and len(text.split()) <= 8:
        situations.append("short post prone to hostile reading")
    return situations


def fallback_explanations(final_dimensions: Dict[str, int], context: str) -> List[str]:
    explanations = []
    if final_dimensions["Aggression"] >= 1:
        explanations.append("The wording lands as harsher than a normal public opinion post, so replies may focus on tone rather than the point.")
    if final_dimensions["Sarcasm / Mockery"] >= 1:
        explanations.append("The sarcastic or dunking tone can make neutral readers read the post as mean-spirited.")
    if final_dimensions["Group Generalization"] >= 1:
        explanations.append("Readers may push back if the post sounds like it is judging a whole group instead of one case.")
    if final_dimensions["Context Inappropriateness"] >= 1:
        explanations.append(f"In a broad public space like {context}, sharp phrasing often attracts criticism faster than in a private chat.")
    if final_dimensions["Misinterpretability"] >= 1:
        explanations.append("Because the statement is short and blunt, people can interpret it more negatively than you intended.")
    return explanations or ["The post is not overtly extreme, but public readers may still react to the tone and framing."]


def merge_risk_scores(heuristic_score: int, llm_score: Optional[int]) -> int:
    if llm_score is None:
        return heuristic_score
    return int(round((0.65 * heuristic_score) + (0.35 * llm_score)))


def run_backlash_risk_pipeline(text: str, context: str = SOCIAL_CONTEXT, audience_profile: str = AUDIENCE_PROFILE) -> PipelineOutput:
    normalized_text = preprocess_text(text)
    tokens = tokenize_text(normalized_text)

    embedder = HybridEmbedder()
    sentence_embedding = embedder.encode_sentence(normalized_text)
    contextual_embedding = embedder.encode_contextual(normalized_text, context, audience_profile)
    scorer = SemanticRiskScorer(embedder)

    heuristic_dimensions = first_pass_risk_classification(
        normalized_text,
        tokens,
        context,
        sentence_embedding,
        contextual_embedding,
        scorer,
    )
    heuristic_risk_score = calculate_weighted_risk_score(heuristic_dimensions)

    llm_prompt = build_qwen_prompt(
        context=context,
        audience_profile=audience_profile,
        original_text=text,
        normalized_text=normalized_text,
        tokens=tokens,
        first_pass_dimensions=heuristic_dimensions,
    )

    analyzer = QwenContextAnalyzer(backend=QWEN_BACKEND)
    llm_response = analyzer.analyze(llm_prompt)
    llm_payload = llm_response["result"]
    llm_backend = llm_response["backend"]
    llm_adjustments = sanitize_llm_adjustments(llm_payload)

    final_dimensions = {
        name: int(np.clip(heuristic_dimensions[name] + llm_adjustments[name], 0, 2))
        for name in DIMENSION_ORDER
    }

    llm_risk_score = None
    if llm_payload and isinstance(llm_payload.get("backlash_probability"), (int, float)):
        llm_risk_score = int(np.clip(round(llm_payload["backlash_probability"]), 0, 100))

    final_risk_score = merge_risk_scores(calculate_weighted_risk_score(final_dimensions), llm_risk_score)
    issue_categories = derive_issue_categories(final_dimensions)
    special_situations = llm_payload.get("special_situations", []) if llm_payload else []
    problematic_phrases = llm_payload.get("problematic_phrases", []) if llm_payload else []
    explanation_points = llm_payload.get("explanations", []) if llm_payload else []
    rewrite_suggestion = llm_payload.get("rewrite", "") if llm_payload else ""

    if not explanation_points:
        explanation_points = fallback_explanations(final_dimensions, context)
    if not problematic_phrases:
        problematic_phrases = detect_problematic_phrases(text)
    if not special_situations:
        special_situations = detect_special_situations(final_dimensions, text, context)
    recommended_context = llm_payload.get("rewrite", "") if llm_payload else ""
    if not recommended_context:
        recommended_context = build_recommended_context(text, final_dimensions, context)
    if not rewrite_suggestion:
        rewrite_suggestion = recommended_context

    sentence_preview = [round(float(value), 4) for value in sentence_embedding[:8]]
    contextual_preview = [round(float(value), 4) for value in contextual_embedding[:8]]

    return PipelineOutput(
        context=context,
        recommended_context=recommended_context,
        audience_profile=audience_profile,
        original_text=text,
        normalized_text=normalized_text,
        tokens=tokens,
        embedding_model=embedder.model_name,
        sentence_embedding_preview=sentence_preview,
        contextual_embedding_preview=contextual_preview,
        heuristic_dimensions=heuristic_dimensions,
        llm_adjustments=llm_adjustments,
        final_dimensions=final_dimensions,
        heuristic_risk_score=heuristic_risk_score,
        llm_risk_score=llm_risk_score,
        final_risk_score=final_risk_score,
        issue_categories=issue_categories,
        special_situations=special_situations,
        problematic_phrases=problematic_phrases,
        explanation_points=explanation_points,
        llm_backend=llm_backend,
        llm_prompt=llm_prompt,
        rewrite_suggestion=rewrite_suggestion,
    )


def display_pipeline_result(result: PipelineOutput) -> None:
    print(f"Input: {result.original_text}")
    print(f"Normalized: {result.normalized_text}")
    print(f"Category: {', '.join(result.issue_categories)}")
    print(f"Context: {result.context}")
    print(f"Recommended Context: {result.recommended_context}")
    print(f"Final Risk Score: {result.final_risk_score}/100")
    print(f"LLM backend: {result.llm_backend}")
    print()
    print("Dimension:")
    for name in DIMENSION_ORDER:
        print(
            f"- {name}: heuristic={result.heuristic_dimensions[name]}, "
            f"adjustment={result.llm_adjustments[name]}, final={result.final_dimensions[name]}"
        )
    print()
    print("Special Situations:")
    if result.special_situations:
        for item in result.special_situations:
            print(f"- {item}")
    else:
        print("- None")
    print()
    print("Problematic Phrases:")
    if result.problematic_phrases:
        for phrase in result.problematic_phrases:
            print(f"- {phrase}")
    else:
        print("- None explicitly highlighted")
    print()
    print("Why readers may criticize it:")
    for reason in result.explanation_points:
        print(f"- {reason}")
    print()
    print(f"Safer rewrite: {result.rewrite_suggestion}")
    print()
    print(f"Runtime device: {RUNTIME_DEVICE}")
    print(f"Detected GPU memory: {GPU_MEMORY_GB} GB")
    print(f"Embedding model: {result.embedding_model}")


## Example Run

If you have a local Ollama server with `qwen3:8b`, keep `QWEN_BACKEND = "auto"` or set `"ollama"`.
This is the safer option on an 8GB GPU because Ollama can use a quantized model and offload to GPU without loading the full Hugging Face checkpoint into RAM.
The direct `transformers` backend is now guarded so it only tries to load when CUDA is available and VRAM is large enough.


In [ ]:
sample_context = "Instagram / Public"
sample_text = "This place is trash lol"

result = run_backlash_risk_pipeline(sample_text, context=sample_context)
display_pipeline_result(result)


In [ ]:
result_table = pd.DataFrame(
    {
        "Category": [', '.join(result.issue_categories)] * len(DIMENSION_ORDER),
        "Context": [result.context] * len(DIMENSION_ORDER),
        "Recommended Context": [result.recommended_context] * len(DIMENSION_ORDER),
        "Dimension": DIMENSION_ORDER,
        "Heuristic": [result.heuristic_dimensions[name] for name in DIMENSION_ORDER],
        "LLM Adjustment": [result.llm_adjustments[name] for name in DIMENSION_ORDER],
        "Final": [result.final_dimensions[name] for name in DIMENSION_ORDER],
        "Description": [DIMENSION_DESCRIPTIONS[name] for name in DIMENSION_ORDER],
    }
)
result_table


In [ ]:
active_dimensions = [name for name, score in result.final_dimensions.items() if score >= 1]
dimension_text = ", ".join(active_dimensions) if active_dimensions else "Low explicit risk"
print(f"Dimension: {dimension_text}")
print(f"Recommended: {result.recommended_context}")
print()
print(result.llm_prompt)


## Notes

- The first pass is not just keyword matching. It combines surface signals with semantic similarity against dimension prototypes.
- The second pass asks Qwen3-8B to correct category scores by `-1 / 0 / +1`, explain likely backlash, and generate a safer rewrite.
- Final score blends weighted category scores with Qwen's backlash probability when a Qwen backend is available.
- `special_situations` is reserved for things like public shaming, target ambiguity, group framing, pile-on risk, or business-review sensitivity.
